In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "# Mongolian medical speech: Colab pipeline\nRuns the whole ASR pipeline from the GitHub repo: data → baseline → Whisper fine-tune → evaluation → TTS demo.\n\n**Before you start**\n1. Runtime → Change runtime type → **T4 GPU**.\n2. Zip your recordings on your Mac (`my_data/wavs` → `wavs.zip`) and upload the zip to Google Drive at `MyDrive/medical-speech/wavs.zip`.\n   The wavs are **not** on GitHub on purpose (privacy, see `my_data/.gitignore`).\n\nEverything that takes long (dataset, checkpoints, final model) is saved to Google Drive, so a disconnect does not lose work."
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 1. GPU check and Google Drive"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "!nvidia-smi\nfrom google.colab import drive\ndrive.mount('/content/drive')\nDRIVE = '/content/drive/MyDrive/medical-speech'\nimport os; os.makedirs(DRIVE, exist_ok=True)"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 2. Get the code from GitHub\nIf the repo is **private**, run the second cell instead: paste a GitHub token (github.com/settings/tokens, *Fine-grained*, read access to this repo only)."
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "REPO = 'https://github.com/Ba1jaa/mongolian-medical-speech.git'\n!rm -rf /content/medical-speech\n!git clone {REPO} /content/medical-speech\n%cd /content/medical-speech\n!ls"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "# Private repo only\nfrom getpass import getpass\ntoken = getpass('GitHub token: ')\n!rm -rf /content/medical-speech\n!git clone https://{token}@github.com/Ba1jaa/mongolian-medical-speech.git /content/medical-speech\ndel token\n%cd /content/medical-speech"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 3. Install packages"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "!apt-get -qq install -y ffmpeg > /dev/null\n!pip install -q -r requirements.txt"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "Optional: log in to Hugging Face (only needed for Common Voice or to upload your model).\nCommon Voice may be skipped even after login; FLEURS + your recordings are enough to start."
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "from huggingface_hub import notebook_login\nnotebook_login()"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 4. Put your recordings in place"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "import zipfile, shutil, os\nos.makedirs('my_data/wavs', exist_ok=True)\nwith zipfile.ZipFile(f'{DRIVE}/wavs.zip') as z:\n    for name in z.namelist():\n        if name.endswith('.wav') and '__MACOSX' not in name:\n            with z.open(name) as src, open(os.path.join('my_data/wavs', os.path.basename(name)), 'wb') as dst:\n                shutil.copyfileobj(src, dst)\nprint(len(os.listdir('my_data/wavs')), 'wav files')"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "Quality check (writes `my_data/metadata.csv` with the files that passed):"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "%cd my_data\n!python check_data.py\n%cd ..\n!wc -l my_data/metadata.csv"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 5. Test the text normalizer"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "!python mn_text.py"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 6. Build the dataset (saved to Drive)\nTakes a while the first time. Re-run only when your data changes."
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "DATA = f'{DRIVE}/data_asr'\n!python 01_prepare_data.py --out \"{DATA}\" --custom_csv my_data/metadata.csv --custom_dir my_data/wavs"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 7. Baselines (before training)\nWrite these numbers down for your report."
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "!python 02_eval_asr.py --model openai/whisper-small --data \"{DATA}\" --split test --max_samples 200\n!python 02_eval_asr.py --model openai/whisper-small --data \"{DATA}\" --split medical_test"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 8. Fine-tune Whisper-small\nSeveral hours on a T4. Checkpoints go to Drive every 1000 steps.\nFor a quick test first, add `--steps 200`. Out of memory → `--batch 4 --accum 4`."
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "OUT = f'{DRIVE}/whisper-small-mn'\n!python 03_finetune_whisper.py --data \"{DATA}\" --out \"{OUT}\""
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 9. Evaluate the fine-tuned model"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "MODEL = f'{OUT}/final'\n!python 02_eval_asr.py --model \"{MODEL}\" --data \"{DATA}\" --split test --max_samples 200\n!python 02_eval_asr.py --model \"{MODEL}\" --data \"{DATA}\" --split medical_test"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "Transcribe one file:"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "import glob\nf = sorted(glob.glob('my_data/wavs/*.wav'))[0]\nprint(f)\n!python 04_transcribe.py --model \"{MODEL}\" --audio \"{f}\""
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 10. TTS baseline (MMS)"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "!python 05_tts_baseline.py --text \"Өвчтөнд өдөрт 2 удаа 5 мг өгнө\" --out hello.wav\nfrom IPython.display import Audio\nAudio('hello.wav')"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": "## 11. Optional: upload the model to Hugging Face (private)"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": "from transformers import WhisperForConditionalGeneration, WhisperProcessor\nHF_REPO = 'Ba1jaa/whisper-small-mn-medical'\nWhisperForConditionalGeneration.from_pretrained(MODEL).push_to_hub(HF_REPO, private=True)\nWhisperProcessor.from_pretrained(MODEL).push_to_hub(HF_REPO, private=True)"
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "provenance": [],
   "gpuType": "T4"
  },
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}